téléchargement de la dataset fusionnée 

In [36]:
import pandas as pd

df = pd.read_csv("cpu_full_dataset.csv")
df.head()


,timestamp,value,serveur_id
0,2014-02-14 14:30:00,0.132,24ae8d
1,2014-02-14 14:35:00,0.134,24ae8d
2,2014-02-14 14:40:00,0.134,24ae8d
3,2014-02-14 14:45:00,0.134,24ae8d
4,2014-02-14 14:50:00,0.134,24ae8d


Exercice A : indexation 
Transformation de la colonne timestamp en format datetime et de la définir comme index du DataFrame afin de pouvoir utiliser les fonctionnalités spécifiques aux séries temporelles

In [37]:

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
df = df.set_index('timestamp')
df.head()


,value,serveur_id
timestamp,,
2014-02-14 14:27:00,51.846,5f5533
2014-02-14 14:30:00,0.132,24ae8d
2014-02-14 14:30:00,1.732,53ea38
2014-02-14 14:32:00,44.508,5f5533
2014-02-14 14:35:00,0.134,24ae8d


Resampling
Regroupement des données par heure et calcul de la moyenne d'utilisation du cpu par heure afin d'obtenir une vision plus globale 

In [38]:
df_hourly = (df.groupby('serveur_id')['value'].resample('H').mean())
df_hourly.head()


C:\Users\21624\AppData\Local\Temp\ipykernel_26864\2023474330.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = (df.groupby('serveur_id')['value'].resample('H').mean())


serveur_id  timestamp          
24ae8d      2014-02-14 14:00:00    0.133667
            2014-02-14 15:00:00    0.122333
            2014-02-14 16:00:00    0.122667
            2014-02-14 17:00:00    0.133667
            2014-02-14 18:00:00    0.128333
Name: value, dtype: float64

Rolling Window
Calcul de la moyenne mobile des 5 derniéres observations afin de mieux observer la tendances d'utilisation du cpu

In [44]:
df['moyenne_mobile_5'] = df.groupby('serveur_id')['value'].transform(lambda x: x.rolling(5).mean())
df.head(100)

,value,serveur_id,moyenne_mobile_5
timestamp,,,
2014-02-14 14:27:00,51.846,5f5533,NaN
2014-02-14 14:30:00,0.132,24ae8d,NaN
2014-02-14 14:30:00,1.732,53ea38,NaN
2014-02-14 14:32:00,44.508,5f5533,NaN
2014-02-14 14:35:00,0.134,24ae8d,NaN
...,...,...,...
2014-02-14 17:05:00,0.136,24ae8d,0.1340
2014-02-14 17:07:00,46.550,5f5533,46.4760
2014-02-14 17:10:00,1.990,53ea38,1.8172


Lagging
Création  d'une variable cible correspondant à la valeur future du CPU, afin de transformer la série temporelle en problème de régression supervisée pour la modélisation prédictive.

In [45]:
df['target'] = df.groupby('serveur_id')['value'].shift(-1)
df.head(100)


,value,serveur_id,moyenne_mobile_5,target
timestamp,,,,
2014-02-14 14:27:00,51.846,5f5533,NaN,44.508
2014-02-14 14:30:00,0.132,24ae8d,NaN,0.134
2014-02-14 14:30:00,1.732,53ea38,NaN,1.732
2014-02-14 14:32:00,44.508,5f5533,NaN,41.244
2014-02-14 14:35:00,0.134,24ae8d,NaN,0.134
...,...,...,...,...
2014-02-14 17:05:00,0.136,24ae8d,0.1340,0.134
2014-02-14 17:07:00,46.550,5f5533,46.4760,48.122
2014-02-14 17:10:00,1.990,53ea38,1.8172,1.738
